# Email handling in Python

## Protocols and formats

### A brief history

Email predates the modern internet. The first network message transfer happened in **1971** on ARPANET (Ray Tomlinson, who also chose `@` as the separator between user and host).

The protocols that still govern email today were standardized through a series of RFCs:

| Year | RFC | What it defined |
|------|-----|-----------------|
| 1982 | [RFC 821](https://www.rfc-editor.org/rfc/rfc821) | **SMTP** – Simple Mail Transfer Protocol (sending) |
| 1982 | [RFC 822](https://www.rfc-editor.org/rfc/rfc822) | **Message format** – headers, body, `From:`, `To:`, `Subject:` |
| 1988 | [RFC 1064](https://www.rfc-editor.org/rfc/rfc1064) → [RFC 3501](https://www.rfc-editor.org/rfc/rfc3501) | **IMAP** – Internet Message Access Protocol (mailbox access) |
| 1991 | [RFC 1225](https://www.rfc-editor.org/rfc/rfc1225) → [RFC 1939](https://www.rfc-editor.org/rfc/rfc1939) | **POP3** – Post Office Protocol v3 (simpler retrieval) |
| 1992–1996 | [RFC 2045–2049](https://www.rfc-editor.org/rfc/rfc2045) | **MIME** – Multipurpose Internet Mail Extensions (attachments, HTML, encodings) |
| 2008 | [RFC 5321](https://www.rfc-editor.org/rfc/rfc5321) | SMTP revised (still current) |
| 2008 | [RFC 5322](https://www.rfc-editor.org/rfc/rfc5322) | Message format revised (still current) |

#### The three-protocol stack

<table>
<thead><tr><th>Your program</th><th>Direction</th><th>Mail server</th><th>Direction</th><th>Recipient's program</th></tr></thead>
<tbody>
<tr><td>compose message</td><td>→ SMTP (port 587/465) →</td><td>stores message in mailbox</td><td>← IMAP (port 993) or POP3 (port 995) ←</td><td>reads mailbox</td></tr>
</tbody>
</table>

- **SMTP** is a *push* protocol: you connect to a server and deliver a message.
- **IMAP** is a *pull* protocol: messages live on the server; you query and manage them remotely.
- **POP3** is a simpler pull protocol that typically downloads and deletes messages; largely superseded by IMAP.

### Message format (RFC 5322 + MIME)

#### The envelope: RFC 5322

An email message is plain text divided into two parts by a blank line:

- **Headers** — a sequence of `Name: value` lines. Some are mandatory (`From`, `To`, `Date`, `Message-ID`); most are optional.
- **Body** — everything after the blank line; in the simplest case, plain text.

A minimal valid message looks like this:

```
From: alice@example.com
To: bob@example.com
Date: Tue, 03 Jun 2025 10:00:00 +0200
Message-ID: <abc123@example.com>
Subject: Hello

Hi Bob, this is the body.
```

Header values that contain non-ASCII characters (accented letters, emoji) must be encoded with **RFC 2047** encoded-words, e.g. `=?utf-8?q?caf=C3=A9?=`. Python handles this transparently.

#### MIME: adding structure to the body

RFC 5322 alone only supports plain text. **MIME** (RFC 2045–2049) extends messages to carry multiple content types — HTML, images, attachments — by splitting the body into **parts**, each with its own headers.

The key header is `Content-Type`. When a message (or a part) contains sub-parts, its type is `multipart/subtype` and a `boundary` string delimits the parts:

```
Content-Type: multipart/mixed; boundary="BOUNDARY"

--BOUNDARY
Content-Type: text/plain

The text body.
--BOUNDARY
Content-Type: application/pdf
Content-Disposition: attachment; filename="report.pdf"
Content-Transfer-Encoding: base64

JVBERi0xLj...
--BOUNDARY--
```

`Content-Transfer-Encoding` (`base64` or `quoted-printable`) handles binary data inside a text-based protocol.

#### The three multipart subtypes

The `multipart` subtype controls how a mail client *interprets* the parts, not how they are serialised (the wire format is the same).

| Subtype | Semantics | Typical use |
|---------|-----------|-------------|
| `multipart/mixed` | Parts are independent items | Body text + file attachments |
| `multipart/alternative` | Parts are alternative renderings of the **same** content; client picks the best one it can display | Plain text + HTML version of the same email |
| `multipart/related` | Parts form a single composite document; the root part references the others by `Content-ID` | HTML body + inline images (`<img src="cid:logo@x">`) |

Real messages often **nest** these types. A common structure for an HTML email with an attachment is:

<table>
<tr><td><b>multipart/mixed</b></td><td>top level</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;└ <b>multipart/alternative</b></td><td>the body in two forms</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├ text/plain</td><td>fallback</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└ multipart/related</td><td>HTML + inline resources</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├ text/html</td><td>the HTML body</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└ image/png (Content-ID: logo)</td><td>inline image</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;└ application/pdf</td><td>attachment</td></tr>
</table>

### Bare hands: raw TCP dialogue

SMTP and IMAP are line-oriented text protocols. To prove it, the two cells below open a plain TCP socket and type every byte manually — no `smtplib`, no `imaplib`, no email libraries of any kind.

The helpers are minimal on purpose:
- `recv()` reads server lines until a non-continuation line arrives and prints each one prefixed with `S:`.
- `send()` adds `\r\n`, sends the bytes, and prints the line prefixed with `C:`.

The SMTP cell sends one message; the IMAP cell immediately retrieves it.

In [1]:
import socket, base64

def send(sock, text):
    print('C:', text)
    sock.sendall((text + '\r\n').encode())

def recv(f, tag=None):
    while True: 
        line = f.readline().decode(errors='replace').rstrip('\r\n')
        print('S:', line)
        if tag is None:
            if len(line) < 4 or line[3] != '-':  # SMTP: stop at non-continuation line
                break
        else:
            if line.startswith(tag):              # IMAP: stop at tagged response
                break

In [11]:
with socket.create_connection(('localhost', 3025)) as sock, sock.makefile('rb') as f:
    recv(f)
    send(sock, 'EHLO notebook');
    recv(f)
    send(sock, 'MAIL FROM:<alice@example.com>');    
    recv(f)
    send(sock, 'RCPT TO:<rec@goo.bar>');
    recv(f)
    send(sock, 'DATA');
    recv(f)
    for line in [
        'From: alice@example.com', 
        'To: rec@goo.bar',
        'Subject: Bare hands', 
        '',
        'No libraries. No frameworks. Just TCP and text.', 
        '.']:
        send(sock, line)
    recv(f)
    send(sock, 'QUIT');

S: 220 /127.0.0.1 GreenMail SMTP Service v2.1.8 ready
C: EHLO notebook
S: 250-/127.0.0.1
S: 250 AUTH PLAIN LOGIN XOAUTH2
C: MAIL FROM:<alice@example.com>
S: 250 OK
C: RCPT TO:<rec@goo.bar>
S: 250 OK
C: DATA
S: 354 Start mail input; end with <CRLF>.<CRLF>
C: From: alice@example.com
C: To: rec@goo.bar
C: Subject: Bare hands
C: 
C: No libraries. No frameworks. Just TCP and text.
C: .
S: 250 OK
C: QUIT


In [ ]:
def cmd(text):
    tag = f'A{next(n):03d}'
    send(sock, f'{tag} {text}')
    recv(f, tag)

with socket.create_connection(('localhost', 3143)) as sock, sock.makefile('rb') as f:
    n = iter(range(1, 100))
    recv(f)
    cmd('LOGIN rec@goo.bar recp')
    cmd('SELECT INBOX')
    cmd('SEARCH ALL')
    cmd('FETCH 1 (BODY.PEEK[HEADER.FIELDS (FROM SUBJECT)])')
    cmd('LOGOUT')

S: * OK IMAP4rev1 Server GreenMail v2.1.8 ready
C: A001 LOGIN rec@goo.bar recp
S: A001 OK LOGIN completed.
C: A002 SELECT INBOX
S: * FLAGS (\Answered \Deleted \Draft \Flagged \Seen)
S: * 4 EXISTS
S: * 0 RECENT
S: * OK [UIDVALIDITY 1780468883]
S: * OK [UIDNEXT 5]
S: * OK No messages unseen
S: * OK [PERMANENTFLAGS (\Answered \Deleted \Draft \Flagged \Seen \*)]
S: A002 OK [READ-WRITE] SELECT completed.
C: A003 SEARCH ALL
S: * SEARCH 1 2 3 4
S: A003 OK SEARCH completed.
C: A004 FETCH 1 (BODY.PEEK[HEADER.FIELDS (FROM SUBJECT)])
S: * 1 FETCH (BODY[HEADER.FIELDS (FROM SUBJECT)] {44}
S: Subject: A simple email
S: From: snd@foo.bar
S: )
S: A004 OK FETCH completed.
C: A005 LOGOUT
S: * BYE IMAP4rev1 Server logging out
S: A005 OK LOGOUT completed.


## Python and email

### Handling messages

Python's standard library includes the [`email`](https://docs.python.org/3/library/email.html) package for constructing, parsing, and manipulating messages entirely in memory — with no network involved.

It has two generations of API:

| API | Class | Policy | Notes |
|-----|-------|--------|-------|
| Legacy (< 3.6) | `email.message.Message` | `compat32` | Low-level, error-prone encoding handling |
| **Modern (≥ 3.6)** | **`email.message.EmailMessage`** | **`default` / `EmailPolicy`** | High-level, Unicode-aware, recommended |

All examples below use the modern API.

#### Creating messages

In [ ]:
from email.message import EmailMessage

# Plain-text message
msg = EmailMessage()
msg['From']    = 'alice@example.com'
msg['To']      = 'bob@example.com'
msg['Subject'] = 'Caffè con accento'   # non-ASCII is handled automatically
msg.set_content('Ciao Bob,\nci vediamo alle 10.\n')

print(msg)   # shows the wire representation

In [ ]:
# HTML alternative: set_content sets the plain-text part,
# add_alternative adds the HTML rendering of the same content.
# Python wraps both in a multipart/alternative automatically.
msg2 = EmailMessage()
msg2['From']    = 'alice@example.com'
msg2['To']      = 'bob@example.com'
msg2['Subject'] = 'HTML email'
msg2.set_content('Ciao Bob,\nci vediamo alle *10*.\n')
msg2.add_alternative(
    '<p>Ciao Bob,<br>ci vediamo alle <strong>10</strong>.</p>',
    subtype='html'
)

print(msg2)

In [ ]:
csv_data = 'name,score\nAlice,95\nBob,87\n'

msg3 = EmailMessage()
msg3['From']    = 'alice@example.com'
msg3['To']      = 'bob@example.com'
msg3['Subject'] = 'Results'
msg3.set_content('See attached.')
# add_attachment promotes the message to multipart/mixed automatically
msg3.add_attachment(csv_data.encode(), maintype='text', subtype='csv', filename='results.csv')

print(msg3)

#### Serialization and deserialization

In [ ]:
import email

# Serialize to str or bytes
raw_str   = str(msg3)          # str — suitable for display
raw_bytes = msg3.as_bytes()    # bytes — what actually goes on the wire

# Deserialize: parse back from bytes (e.g. as retrieved from IMAP)
parsed = email.message_from_bytes(raw_bytes, policy=email.policy.default)
print(f'Content-Type : {parsed.get_content_type()}')
print(f'Subject      : {parsed["Subject"]}')

# Walk all parts
for part in parsed.walk():
    ct  = part.get_content_type()
    cd  = part.get_content_disposition()
    print(f'  part: {ct:30s}  disposition={cd}')

In [ ]:
# The modern API also offers typed accessors — no need to manually decode payloads
for part in parsed.walk():
    cd = part.get_content_disposition()
    if cd == 'attachment':
        print(f'Attachment: {part.get_filename()}')
        print(part.get_content().decode())   # returns decoded str/bytes directly
    elif part.get_content_type() == 'text/plain':
        print(f'Plain text body: {part.get_content()!r}')

### Sending mail with `smtplib`

`smtplib` implements the SMTP client side. The typical exchange with a server is:

1. TCP connect
2. Server greets with `220`
3. Client sends `EHLO` — server replies with its capabilities
4. (optional) Upgrade to TLS with `STARTTLS`
5. `AUTH` — authenticate
6. `MAIL FROM` / `RCPT TO` / `DATA` — deliver the message
7. `QUIT`

Python's `smtplib.SMTP` handles all of this. The three port conventions are:

| Port | Name | TLS |
|------|------|-----|
| 25 | SMTP (server-to-server relay) | none / STARTTLS |
| 587 | **Submission** (client → server) | STARTTLS (call `starttls()` after connect) |
| 465 | **SMTPS** (client → server) | TLS from the start — use `SMTP_SSL` |

#### Basic usage

`SMTP` works as a context manager: the connection is closed (and `QUIT` is sent) on exit.

In [ ]:
import smtplib
from email.message import EmailMessage

msg = EmailMessage()
msg['From']    = 'alice@example.com'
msg['To']      = 'bob@example.com'
msg['Subject'] = 'Hello from smtplib'
msg.set_content('This is a test.')

# Port 3025 is the GreenMail test server used in this lab (no TLS needed)
with smtplib.SMTP('localhost', 3025) as smtp:
    smtp.login('alice@example.com', 'alice')
    smtp.send_message(msg)   # reads From/To/Cc headers automatically

#### Against a real server (STARTTLS on port 587)

```python
with smtplib.SMTP('smtp.gmail.com', 587) as smtp:
    smtp.starttls()                          # upgrade to TLS before auth
    smtp.login('you@gmail.com', app_password)
    smtp.send_message(msg)
```

#### Against a real server (implicit TLS on port 465)

```python
import ssl

with smtplib.SMTP_SSL('smtp.gmail.com', 465, context=ssl.create_default_context()) as smtp:
    smtp.login('you@gmail.com', app_password)
    smtp.send_message(msg)
```

> **Note:** Gmail and most providers require an *app password* (not your account password) when using SMTP directly.

#### Debugging: see the SMTP dialogue

Set `smtp.set_debuglevel(1)` to print every command and response — useful when a server rejects your message and the error code is unclear.

In [ ]:
with smtplib.SMTP('localhost', 3025) as smtp:
    smtp.set_debuglevel(1)
    smtp.login('alice@example.com', 'alice')
    smtp.send_message(msg)

### Receiving mail with `imaplib`

IMAP is a *stateful* protocol: after connecting you are always in one of several states (not-authenticated → authenticated → selected). Most operations only make sense in the *selected* state (after `SELECT`ing a mailbox).

Key concepts:

| Concept | Meaning |
|---------|---------|
| **Mailbox** | A folder on the server (`INBOX`, `Sent`, `Drafts`, …) |
| **Sequence number** | Position of a message in the currently selected mailbox — changes as messages are added/deleted |
| **UID** | Stable numeric identifier for a message — survives expunge and reconnects |
| **Flag** | Per-message state: `\Seen`, `\Answered`, `\Flagged`, `\Deleted`, `\Draft` |
| **SEARCH** | Server-side filtering by date, flag, header, size, … |
| **FETCH** | Retrieve part or all of a message (`RFC822` = full wire format) |

Fetching a message does **not** mark it as `\Seen` unless you explicitly fetch the `BODY[]` (without `.PEEK`). Using `BODY.PEEK[]` is safe for read-only inspection.

#### Basic usage with `imaplib`

In [ ]:
import imaplib, email

with imaplib.IMAP4('localhost', 3143) as imap:
    imap.login('bob@example.com', 'bob')

    # List available mailboxes
    status, mailboxes = imap.list()
    for mb in mailboxes:
        print(mb.decode())

In [ ]:
with imaplib.IMAP4('localhost', 3143) as imap:
    imap.login('bob@example.com', 'bob')
    imap.select('INBOX')   # SELECT enters the selected state; use EXAMINE for read-only

    # Search returns a space-separated list of sequence numbers
    status, data = imap.search(None, 'ALL')
    seq_nums = data[0].split()
    print(f'{len(seq_nums)} message(s) in INBOX')

    for num in seq_nums:
        # BODY.PEEK[RFC822] fetches the full message without marking it \Seen
        _, msg_data = imap.fetch(num, '(BODY.PEEK[RFC822])')
        raw = msg_data[0][1]
        msg = email.message_from_bytes(raw, policy=email.policy.default)
        print(f'  [{num.decode()}] {msg["Subject"]}  flags=', end='')

        # Fetch flags separately
        _, flag_data = imap.fetch(num, '(FLAGS)')
        print(flag_data[0].decode())

#### Searching and flagging

`SEARCH` accepts a rich set of criteria that the server evaluates — no need to download messages first.

In [ ]:
with imaplib.IMAP4('localhost', 3143) as imap:
    imap.login('bob@example.com', 'bob')
    imap.select('INBOX')

    _, unseen = imap.search(None, 'UNSEEN')
    print('Unseen:', unseen[0].decode())

    _, from_alice = imap.search(None, 'FROM', 'alice@example.com')
    print('From alice:', from_alice[0].decode())

    # Mark a message as seen
    if unseen[0]:
        first = unseen[0].split()[0]
        imap.store(first, '+FLAGS', r'\Seen')

    # Delete: mark \Deleted then expunge
    # imap.store(num, '+FLAGS', r'\Deleted')
    # imap.expunge()

#### A higher-level alternative: `imap-tools`

`imaplib` is complete but low-level: responses are raw bytes, search criteria are stringly-typed, and you have to parse everything yourself. The third-party [`imap-tools`](https://github.com/ikvk/imap_tools) library wraps it with a clean, Pythonic API.

| Task | `imaplib` | `imap-tools` |
|------|-----------|--------------|
| Fetch all messages | `search` + `fetch` loop | `mailbox.fetch()` iterator |
| Read subject | `email.message_from_bytes(raw)['Subject']` | `msg.subject` |
| List attachments | walk parts, filter by disposition | `msg.attachments` |
| Filter server-side | string criteria | `AND(from_='alice@…', seen=False)` |

In [ ]:
from imap_tools import MailBoxUnencrypted, AND

with MailBoxUnencrypted('localhost', 3143).login('bob@example.com', 'bob') as mailbox:
    for msg in mailbox.fetch(AND(from_='alice@example.com')):
        print(f'Subject : {msg.subject}')
        print(f'Date    : {msg.date}')
        print(f'Text    : {msg.text[:60]!r}')
        for att in msg.attachments:
            print(f'  attachment: {att.filename} ({len(att.payload)} bytes)')